In [46]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
import uuid
from datetime import datetime

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 48, Finished, Available, Finished, False)

In [47]:
BRONZE_TABLE = f"LH_Bronze.dbo.bronze_passenger_loading"
SILVER_REFRESH_QUEUE = f"LH_Silver.dbo.silver_refresh_queue"
SILVER_GTR_PASSENGER_LOADING = "LH_Silver.dbo.silver_gtr_passenger_loading"
GOLD_REFRESH_QUEUE = f"LH_Gold.dbo.gold_refresh_queue"

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 49, Finished, Available, Finished, False)

In [48]:
queue_df = spark.read.table(SILVER_REFRESH_QUEUE).where(col("status") == "PENDING")
display(queue_df)

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c224dc13-1501-4ace-92f6-1c22848a1bf4)

In [49]:
# creating the silver_gtr_passenger_loading table if missing.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_GTR_PASSENGER_LOADING}
USING DELTA
AS
SELECT 
    *,
    CAST(NULL AS STRING) AS queue_id,
    CAST(NULL AS TIMESTAMP) AS silver_load_timestamp
FROM {BRONZE_TABLE}
WHERE 1 = 0
""")

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 51, Finished, Available, Finished, False)

DataFrame[]

In [50]:
def queue_gold_refresh(rail_period, silver_load_timestamp):

    # Remove any existing active queue entries for this rail period
    spark.sql(f"""
        DELETE FROM {GOLD_REFRESH_QUEUE}
        WHERE rail_period = '{rail_period}'
          AND status IN ('PENDING', 'FAILED')
    """)
    
    queue_schema = StructType([
        StructField("queue_id", StringType(), False),
        StructField("rail_period", StringType(), False),
        StructField("silver_load_timestamp", TimestampType(), False),
        StructField("processing_start_timestamp", TimestampType(), True),
        StructField("processing_end_timestamp", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("error_message", StringType(), True)
    ])

    queue_data = [{
        "queue_id": str(uuid.uuid4()),
        "rail_period": str(rail_period),
        "silver_load_timestamp": silver_load_timestamp,
        "processing_start_timestamp": None,
        "processing_end_timestamp": None,
        "status": "PENDING",
        "error_message": None
    }]

    queue_df = spark.createDataFrame(
        queue_data,
        schema=queue_schema
    )

    queue_df.write \
        .mode("append") \
        .saveAsTable(GOLD_REFRESH_QUEUE)

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 52, Finished, Available, Finished, False)

In [51]:
from datetime import datetime

for row in queue_df.collect():

    queue_id = row["queue_id"]
    rail_period = row["rail_period"]

    print(f"Processing Rail Period: {rail_period}")

    try:

        print("Mark PROCESSING")
        spark.sql(f"""
        UPDATE {SILVER_REFRESH_QUEUE}
        SET
            status = 'PROCESSING',
            processing_start_timestamp = current_timestamp()
        WHERE queue_id = '{queue_id}'
        """)

        print("Read Bronze")
        bronze_df = (
            spark.table(BRONZE_TABLE)
                 .filter(col("rail_period") == rail_period)
        )

        # Capture a single timestamp for this load
        silver_load_timestamp = datetime.now()

        print("Add Silver metadata")
        silver_df = (
            bronze_df
                .withColumn("queue_id", lit(queue_id))
                .withColumn(
                    "silver_load_timestamp",
                    lit(silver_load_timestamp)
                )
        )

        print("Delete existing Silver data")
        spark.sql(f"""
        DELETE FROM {SILVER_GTR_PASSENGER_LOADING}
        WHERE rail_period = '{rail_period}'
        """)

        print("Write Silver data")
        silver_df.write \
            .mode("append") \
            .saveAsTable(SILVER_GTR_PASSENGER_LOADING)

        print("Create Gold refresh queue")
        queue_gold_refresh(
            rail_period,
            silver_load_timestamp
        )

        print("Mark COMPLETED")
        spark.sql(f"""
        UPDATE {SILVER_REFRESH_QUEUE}
        SET
            status = 'COMPLETED',
            processing_end_timestamp = current_timestamp(),
            error_message = NULL
        WHERE queue_id = '{queue_id}'
        """)

    except Exception as e:

        error_msg = str(e).replace("'", "''")

        spark.sql(f"""
        UPDATE {SILVER_REFRESH_QUEUE}
        SET
            status = 'FAILED',
            processing_end_timestamp = current_timestamp(),
            error_message = '{error_msg[:500]}'
        WHERE queue_id = '{queue_id}'
        """)

        raise

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 53, Finished, Available, Finished, False)

Processing Rail Period: 2705
Mark PROCESSING
Read Bronze
Add Silver metadata
Delete existing Silver data
Write Silver data
Create Gold refresh queue
Mark COMPLETED
Processing Rail Period: 2706
Mark PROCESSING
Read Bronze
Add Silver metadata
Delete existing Silver data
Write Silver data
Create Gold refresh queue
Mark COMPLETED
Processing Rail Period: 2704
Mark PROCESSING
Read Bronze
Add Silver metadata
Delete existing Silver data
Write Silver data
Create Gold refresh queue
Mark COMPLETED


In [52]:
df = spark.read.table(SILVER_GTR_PASSENGER_LOADING)
display(df)

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 71f961ae-fb60-461a-8849-4b49fa90406e)

In [53]:
# reset the queue for testing
# spark.sql(f"""
# UPDATE {SILVER_REFRESH_QUEUE}
# SET status = 'PENDING'
# ,   error_message = NULL
# """)

# spark.sql(f"DELETE FROM {GOLD_REFRESH_QUEUE}")

silver_refresh_queue_df = spark.read.table(SILVER_REFRESH_QUEUE)
gold_refresh_queue_df = spark.read.table(GOLD_REFRESH_QUEUE)

display(silver_refresh_queue_df)
display(gold_refresh_queue_df)

StatementMeta(, 49c06ed4-c815-4b72-9288-515aac63acab, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d02dc358-1524-452f-8464-3a619f8b2698)

SynapseWidget(Synapse.DataFrame, 78126bf5-ad6a-45c5-88c6-33af5e0fa658)